In [0]:
# 02_silver_transform.py
# Notebook: 02_silver_transform
# Ejecutar en Databricks (Python)

from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()

# *******************************************************************
# AJUSTE DE RUTAS A UNITY CATALOG VOLUMES
# *******************************************************************
CATALOG_NAME = "olist"
SCHEMA_NAME = "olist_csv"
BRONZE_VOLUME_NAME = "bronce_data"
SILVER_VOLUME_NAME = "silver_data"

# Rutas de Origen y Destino
bronze_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{BRONZE_VOLUME_NAME}/"
silver_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{SILVER_VOLUME_NAME}/"

# *******************************************************************
# PASO 1: Crear el Volume de destino (Silver) si no existe
# *******************************************************************
try:
    print(f"Verificando y creando el Volume de destino Silver: {CATALOG_NAME}.{SCHEMA_NAME}.{SILVER_VOLUME_NAME}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{SILVER_VOLUME_NAME}")
    print("Volume Silver verificado/creado exitosamente.")
except Exception as e:
    print(f"ERROR: No se pudo crear el Volume Silver. Verifica tus permisos de Unity Catalog.")
    print(f"Detalle del error de creación: {e}")
    raise # Detener la ejecución si el destino no se puede asegurar

# *******************************************************************
# PASO 2: Cargar todas las tablas bronze (incluyendo las nuevas)
# *******************************************************************
print("Cargando tablas de la capa Bronze...")
customers = spark.read.parquet(bronze_path + "customers")
orders = spark.read.parquet(bronze_path + "orders")
order_items = spark.read.parquet(bronze_path + "order_items")
order_payments = spark.read.parquet(bronze_path + "order_payments")
order_reviews = spark.read.parquet(bronze_path + "order_reviews")
products = spark.read.parquet(bronze_path + "products")
sellers = spark.read.parquet(bronze_path + "sellers")

# Archivos adicionales
geolocation = spark.read.parquet(bronze_path + "geolocation")
translation = spark.read.parquet(bronze_path + "translation")
premium_flag = spark.read.parquet(bronze_path + "premium_flag")


# *******************************************************************
# PASO 3: Transformaciones (Convertir tipos de datos y agregaciones)
# *******************************************************************

# CORRECCIÓN DE ERROR [CANNOT_PARSE_TIMESTAMP]: Se usa F.try_to_timestamp.
# CORRECCIÓN DE ERROR [UNRESOLVED_COLUMN]: Se envuelve la variable DATE_FORMAT en F.lit().
DATE_FORMAT = 'yyyy-MM-dd HH:mm:ss'
DATE_FORMAT_LITERAL = F.lit(DATE_FORMAT) # Definimos el literal de Spark una vez

print("Realizando transformaciones de fechas y agregaciones...")
orders = orders \
    .withColumn("order_purchase_timestamp", F.try_to_timestamp("order_purchase_timestamp", DATE_FORMAT_LITERAL)) \
    .withColumn("order_approved_at", F.try_to_timestamp("order_approved_at", DATE_FORMAT_LITERAL)) \
    .withColumn("order_delivered_customer_date", F.try_to_timestamp("order_delivered_customer_date", DATE_FORMAT_LITERAL)) \
    .withColumn("order_estimated_delivery_date", F.try_to_timestamp("order_estimated_delivery_date", DATE_FORMAT_LITERAL))

# Aplicar formato explícito y try_to_timestamp a order_reviews
order_reviews = order_reviews \
    .withColumn("review_creation_date", F.try_to_timestamp("review_creation_date", DATE_FORMAT_LITERAL)) \
    .withColumn("review_answer_timestamp", F.try_to_timestamp("review_answer_timestamp", DATE_FORMAT_LITERAL))

# Agregar resumen por order_id (suma de items, conteos)
order_items_agg = order_items.groupBy("order_id").agg(
    F.sum(F.col("price")).alias("order_sum_price"),
    F.sum(F.col("freight_value")).alias("order_sum_freight"),
    F.count("*").alias("items_count"),
    F.countDistinct("product_id").alias("distinct_products")
)

# Agregar pagos por order
payments_agg = order_payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("payment_sum"),
    F.avg("payment_installments").alias("avg_installments"),
    F.countDistinct("payment_type").alias("n_payment_types")
)

# Join: orders + items + payments + reviews (left joins para no perder orders)
orders_full = orders.join(order_items_agg, on="order_id", how="left") \
                    .join(payments_agg, on="order_id", how="left") \
                    .join(order_reviews.select("order_id", "review_score", "review_comment_message", "review_creation_date"), on="order_id", how="left")

# *******************************************************************
# PASO 4: Guardar Silver (Incluyendo las nuevas tablas no transformadas)
# *******************************************************************
print("Guardando tablas en la capa Silver...")
customers.write.mode("overwrite").parquet(silver_path + "customers")
orders_full.write.mode("overwrite").parquet(silver_path + "orders_full")
order_items.write.mode("overwrite").parquet(silver_path + "order_items")
payments_agg.write.mode("overwrite").parquet(silver_path + "order_payments_agg")
products.write.mode("overwrite").parquet(silver_path + "products")
sellers.write.mode("overwrite").parquet(silver_path + "sellers")

# Guardar los archivos adicionales en Silver, tal cual (sin transformaciones en esta etapa)
geolocation.write.mode("overwrite").parquet(silver_path + "geolocation")
translation.write.mode("overwrite").parquet(silver_path + "translation")
premium_flag.write.mode("overwrite").parquet(silver_path + "premium_flag")


print("Transformación Silver completada.")
